In [10]:
!nvidia-smi

Mon Mar 31 17:30:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080        Off |   00000000:21:00.0  On |                  N/A |
|  0%   42C    P8             21W /  320W |       4MiB /  10240MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
pip install sacremoses

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [24]:
# === Flamingo-Style Radiology Report Generator (Trainable, BioGPT) ===
# BioGPT + Projected CheXNet Feature Maps + Gated Cross-Attention + BLEU Eval

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.model_selection import train_test_split

# === 1. Load CheXNet Feature Maps ===
def load_features(pkl_path):
    with open(pkl_path, "rb") as f:
        return pickle.load(f)

# === 2. Load Text Data ===
def load_text_data(csv_path):
    df = pd.read_csv(csv_path)
    df.fillna("", inplace=True)
    df['image_id'] = df['image_id'].apply(lambda x: x if x.endswith(".png") else x + ".png")
    df['prompt'] = "FINDINGS: " + df['findings'].astype(str).str.strip() + "\nIMPRESSION:"
    df['target'] = df['impression'].astype(str).str.strip()
    return df

# === 3. Projector (1024 → match LM dim) ===
class VisualProjector(nn.Module):
    def __init__(self, input_dim=1024, output_dim=1024):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

# === 4. Gated Cross-Attention ===
class GatedCrossAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)
        self.gate = nn.Parameter(torch.tensor(0.1))

    def forward(self, text_embeds, visual_embeds):
        attn_out, _ = self.attn(text_embeds, visual_embeds, visual_embeds)
        return text_embeds + self.gate * attn_out

# === 5. BioGPT Wrapper with Attention ===
class FlamingoBioGPTWrapper(nn.Module):
    def __init__(self, model_name="microsoft/BioGPT"):
        super().__init__()
        self.gpt2 = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        hidden_size = self.gpt2.config.hidden_size
        self.projector = VisualProjector(input_dim=1024, output_dim=hidden_size)
        self.cross_attn = GatedCrossAttention(embed_dim=hidden_size)

    def forward(self, visual_feats, input_ids, attention_mask):
        text_embeds = self.gpt2.get_input_embeddings()(input_ids)
        visual_embeds = self.projector(visual_feats)
        attended = self.cross_attn(text_embeds, visual_embeds)
        outputs = self.gpt2(inputs_embeds=attended, attention_mask=attention_mask, labels=input_ids)
        return outputs.loss

    def generate_report(self, visual_feats, prompt, max_new_tokens=80):
        input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(visual_feats.device)
        visual_feats = visual_feats.unsqueeze(0) if visual_feats.ndim == 2 else visual_feats
        visual_embeds = self.projector(visual_feats)
        text_embeds = self.gpt2.get_input_embeddings()(input_ids)
        attended = self.cross_attn(text_embeds, visual_embeds)
        output_ids = self.gpt2.generate(
            inputs_embeds=attended,
            max_new_tokens=max_new_tokens,
            pad_token_id=self.tokenizer.eos_token_id,
            do_sample=False
        )
        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

# === 6. Dataset + Training Logic ===
class ReportDataset(Dataset):
    def __init__(self, df, features, tokenizer, max_length=128):
        self.df = df
        self.features = features
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids = self.tokenizer(row['prompt'] + ' ' + row['target'], max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")
        image_feat = torch.tensor(self.features[row['image_id']], dtype=torch.float32)  # (49, 1024)
        return image_feat, input_ids['input_ids'].squeeze(0), input_ids['attention_mask'].squeeze(0)

# === 7. BLEU Evaluation ===    
def evaluate_bleu(model, df, features, tokenizer, num_samples=None):
    smoothie = SmoothingFunction().method4
    scores = []
    model.eval()

    # Set total safely
    total = len(df) if num_samples is None else min(len(df), num_samples)

    for i in range(total):
        row = df.iloc[i]
        feat = torch.tensor(features[row['image_id']], dtype=torch.float32).to(next(model.parameters()).device)
        generated = model.generate_report(feat, row['prompt'])
        reference = row['target']
        score = sentence_bleu([reference.split()], generated.split(), smoothing_function=smoothie)
        scores.append(score)

    return np.mean(scores)


# === 8. Main Training Loop ===
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    FEATURES_PATH = "/home/vkukk2/Desktop/Vaishnavi/chestrays/feature_maps.pkl"
    CSV_PATH = "/home/vkukk2/Desktop/Vaishnavi/chestrays/report_training_data.csv"

    features = load_features(FEATURES_PATH)
    df = load_text_data(CSV_PATH)
    df = df[df['image_id'].isin(features.keys())].reset_index(drop=True)

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    model = FlamingoBioGPTWrapper("microsoft/BioGPT")
    tokenizer = model.tokenizer
    model.to(device)

    train_dataset = ReportDataset(train_df, features, tokenizer)
    val_dataset = ReportDataset(val_df, features, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

    # === Phase 1: Freeze BioGPT ===
    for param in model.gpt2.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
    best_bleu = 0.0

    for epoch in range(20):
        print(f"\nEpoch {epoch+1}/20")
        model.train()
        total_loss = 0
        for visual_feats, input_ids, attn_mask in tqdm(train_loader):
            visual_feats, input_ids, attn_mask = visual_feats.to(device), input_ids.to(device), attn_mask.to(device)
            loss = model(visual_feats, input_ids, attn_mask)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")

        #train_bleu = evaluate_bleu(model, train_df, features, tokenizer)
        #val_bleu = evaluate_bleu(model, val_df, features, tokenizer)
       # print(f"Train BLEU: {train_bleu:.4f} | Validation BLEU: {val_bleu:.4f}")

        #if val_bleu > best_bleu:
            #best_bleu = val_bleu
            #torch.save(model.state_dict(), "best_flamingo_biogpt.pt")
            #print("Best model saved.")

    # === Phase 2: Unfreeze BioGPT ===
    for param in model.gpt2.parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    for epoch in range(50):
        print(f"\nPhase 2 Epoch {epoch+1}/50")
        model.train()
        total_loss = 0
        for visual_feats, input_ids, attn_mask in tqdm(train_loader):
            visual_feats, input_ids, attn_mask = visual_feats.to(device), input_ids.to(device), attn_mask.to(device)
            loss = model(visual_feats, input_ids, attn_mask)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")

# Save model after all epochs (no indent)
torch.save(model.state_dict(), "flamingo_final.pt")
print("\n Final model saved to flamingo_final.pt")

# Evaluate (no indent)
model.load_state_dict(torch.load("flamingo_final.pt"))
model.eval()
bleu = evaluate_bleu(model, val_df, features, tokenizer)
print(f"\n Final Average BLEU Score on Validation Set: {bleu:.4f}")

# === Run a sample generation ===
sample_idx = 0
row = val_df.iloc[sample_idx]
image_feat = torch.tensor(features[row['image_id']], dtype=torch.float32).to(device)
print("\n Prompt:\n", row['prompt'])
print(" Ground Truth:\n", row['target'])
print(" Generated Report:\n", model.generate_report(image_feat, row['prompt']))




Epoch 1/20


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2988/2988 [01:52<00:00, 26.57it/s]


Epoch 1 Train Loss: 1.3444

Epoch 2/20


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2988/2988 [01:53<00:00, 26.22it/s]


Epoch 2 Train Loss: 1.0393

Epoch 3/20


...

Epoch 48 Train Loss: 0.0699

Phase 2 Epoch 49/50


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2988/2988 [04:49<00:00, 10.31it/s]


Epoch 49 Train Loss: 0.0698

Phase 2 Epoch 50/50


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2988/2988 [04:50<00:00, 10.29it/s]


Epoch 50 Train Loss: 0.0697

 Final model saved to flamingo_final.pt

 Final Average BLEU Score on Validation Set: 0.5386

 Prompt:
 FINDINGS: The lungs are clear. There are calcified left hilar lymph XXXX. The heart and mediastinum are normal. The skeletal structures are notable for an old apparent fracture at T12-L1 or congenital fusion unchanged from the prior study.
IMPRESSION:
 Ground Truth:
 1. No acute pulmonary disease. 2. Possible old injury or developmental anomaly partially T12-L1.
 Generated Report:
 1. No acute pulmonary disease.


In [26]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import random

# === Load best model for inference ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FlamingoBioGPTWrapper("microsoft/BioGPT").to(device)
model.load_state_dict(torch.load("flamingo_final.pt", map_location=device))
model.eval()

# === BLEU smoothing ===
smoothie = SmoothingFunction().method4

# === Select 5 random validation samples ===
sampled_val = val_df.sample(n=5, random_state=42).reset_index(drop=True)

# === Inference and evaluation ===
for i in range(5):
    row = sampled_val.iloc[i]
    image_id = row['image_id']
    prompt = row['prompt']
    reference = row['target']

    visual_feat = torch.tensor(features[image_id], dtype=torch.float32).unsqueeze(0).to(device)
    generated = model.generate_report(visual_feat.squeeze(0), prompt)

    # Compute BLEU score
    bleu = sentence_bleu([reference.split()], generated.split(), smoothing_function=smoothie)

    # Print results
    print(f"\n Sample {i+1}")
    print(" Image ID:", image_id)
    print(" Prompt:\n", prompt)
    print(" Ground Truth:\n", reference)
    print(" Generated Report:\n", generated)
    print(f" BLEU Score: {bleu:.4f}")



 Sample 1
 Image ID: CXR2740_IM-1195-2001.png
 Prompt:
 FINDINGS: The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities. Lateral view reveals degenerative changes of the thoracic spine.
IMPRESSION:
 Ground Truth:
 No acute cardiopulmonary abnormalities. .
 Generated Report:
 No acute cardiopulmonary abnormalities.
 BLEU Score: 0.7788

 Sample 2
 Image ID: CXR814_IM-2345-1001.png
 Prompt:
 FINDINGS: There is a calcified granuloma in the lateral left base. There is no pleural effusion or pneumothorax. The heart is not significantly enlarged. There are calcified left hilar lymph XXXX. There are atherosclerotic changes of the aorta. Arthritic changes of the skeletal structures are noted as well as scoliosis and lumbar region.
IMPRESSION:
 Ground Truth:
 Old granulomatous disease and senescent changes but no acute pulmon

In [14]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import random

# === Load best model for inference ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FlamingoBioGPTWrapper("microsoft/BioGPT").to(device)
model.load_state_dict(torch.load("best_flamingo_biogpt.pt", map_location=device))
model.eval()

# === BLEU smoothing ===
smoothie = SmoothingFunction().method4

# === Select 10 random validation samples ===
sampled_val = val_df.sample(n=5, random_state=42).reset_index(drop=True)

# === Inference and evaluation ===
for i in range(10):
    row = sampled_val.iloc[i]
    image_id = row['image_id']
    prompt = row['prompt']
    reference = row['target']

    visual_feat = torch.tensor(features[image_id], dtype=torch.float32).unsqueeze(0).to(device)
    generated = model.generate_report(visual_feat.squeeze(0), prompt)

    # Compute BLEU score
    bleu = sentence_bleu([reference.split()], generated.split(), smoothing_function=smoothie)

    # Print results
    print(f"\n Sample {i+1}")
    print(" Image ID:", image_id)
    print(" Prompt:\n", prompt)
    print(" Ground Truth:\n", reference)
    print(" Generated Report:\n", generated)
    print(f" BLEU Score: {bleu:.4f}")



 Sample 1
 Image ID: CXR2740_IM-1195-2001.png
 Prompt:
 FINDINGS: The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities. Lateral view reveals degenerative changes of the thoracic spine.
IMPRESSION:
 Ground Truth:
 No acute cardiopulmonary abnormalities. .
 Generated Report:
 No acute cardiopulmonary abnormalities.
 BLEU Score: 0.7788

 Sample 2
 Image ID: CXR814_IM-2345-1001.png
 Prompt:
 FINDINGS: There is a calcified granuloma in the lateral left base. There is no pleural effusion or pneumothorax. The heart is not significantly enlarged. There are calcified left hilar lymph XXXX. There are atherosclerotic changes of the aorta. Arthritic changes of the skeletal structures are noted as well as scoliosis and lumbar region.
IMPRESSION:
 Ground Truth:
 Old granulomatous disease and senescent changes but no acute pulmon